# 02 — Build the project datasets

This notebook turns the raw Food.com files into the smaller, cleaned datasets the team works with:

| File | What it is | Used for |
|---|---|---|
| `data/processed/recipes.csv` | every recipe in one of the 9 shortlisted cuisines, with **one** simple `cuisine` label | networks |
| `data/processed/reviews.csv` | the cleaned reviews of exactly those recipes | NLP |
| `data/processed/recipes_mini.csv` | a random 30% of the recipes of **each** cuisine (70% removed) | networks, when the full set is too slow |
| `data/processed/reviews_mini.csv` | the reviews of the mini recipes | NLP on the mini set |

The cuisine shortlist and the problems we clean here come from `01_initial_eda.ipynb`. Every cleaning step prints how many rows it removes, so nothing disappears silently.

## 0. Setup

In [1]:
import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

SHORTLIST = ["italian", "mexican", "indian", "greek", "french", "chinese", "middle-eastern", "thai", "japanese"]
MINI_KEEP_FRACTION = 0.30   # keep 30% of each cuisine = randomly remove 70%
RNG_SEED = 42
pd.set_option("display.max_colwidth", 90)

## 1. Load and parse the raw files
As in notebook 01: the list columns are stored as text and parsed with `ast.literal_eval`. We keep the original row number of each review as `review_id`, so any review can be traced back to `RAW_interactions.csv`.

In [2]:
recipes_raw = pd.read_csv(RAW / "RAW_recipes.csv", parse_dates=["submitted"])
reviews_raw = pd.read_csv(RAW / "RAW_interactions.csv", parse_dates=["date"])
reviews_raw = reviews_raw.rename_axis("review_id").reset_index()

for col in ["tags", "ingredients", "steps", "nutrition"]:
    recipes_raw[col] = recipes_raw[col].map(ast.literal_eval)

print(f"raw recipes: {len(recipes_raw):,} | raw reviews: {len(reviews_raw):,}")

raw recipes: 231,637 | raw reviews: 1,132,367


## 2. One simple cuisine label per recipe
A recipe belongs to a cuisine if it carries that tag. Food.com also has sub-cuisine tags (`szechuan`, `cantonese`, `baja`, `oaxacan`, `lebanese`, `turkish`, ...). Notebook 01 found which tags sit *under* each shortlisted cuisine. The check below shows that every recipe with such a sub-tag also has the main tag. So looking only at the main tag loses nothing, and the label is always the simple one: a `szechuan` recipe is just `chinese`.

In [3]:
SUBTAGS = {
    "chinese": ["szechuan", "cantonese", "hunan", "beijing", "mongolian"],
    "mexican": ["baja", "oaxacan"],
    "middle-eastern": ["lebanese", "turkish", "iranian-persian", "jewish-sephardi", "saudi-arabian",
                       "iraqi", "palestinian", "georgian"],
}
check = pd.DataFrame([
    {"main tag": main, "sub-tag": sub,
     "recipes with sub-tag": recipes_raw["tags"].map(lambda t, s=sub: s in t).sum(),
     "...without the main tag": recipes_raw["tags"].map(lambda t, s=sub, m=main: s in t and m not in t).sum()}
    for main, subs in SUBTAGS.items() for sub in subs
])
display(check)
assert check["...without the main tag"].sum() == 0

,main tag,sub-tag,recipes with sub-tag,...without the main tag
0,chinese,szechuan,114,0
1,chinese,cantonese,90,0
2,chinese,hunan,31,0
3,chinese,beijing,18,0
4,chinese,mongolian,13,0
5,mexican,baja,160,0
6,mexican,oaxacan,72,0
7,middle-eastern,lebanese,308,0
8,middle-eastern,turkish,286,0
9,middle-eastern,iranian-persian,252,0


Recipes tagged with **two** shortlisted cuisines (for example `italian` and `french`) are dropped: they would need two labels, and a network or text comparison between cuisines is cleaner when each recipe counts for exactly one cuisine.

In [4]:
in_cuisine = pd.DataFrame({c: recipes_raw["tags"].map(lambda t, c=c: c in t) for c in SHORTLIST})
n_cuisines = in_cuisine.sum(axis=1)

recipes = recipes_raw[n_cuisines == 1].copy()
recipes["cuisine"] = in_cuisine[n_cuisines == 1].idxmax(axis=1)

log = []  # (step, recipes removed, recipes left) for the summary at the end
log.append(("recipes in a shortlisted cuisine", None, int((n_cuisines >= 1).sum())))
log.append(("drop recipes in two or more shortlisted cuisines", int((n_cuisines >= 2).sum()), len(recipes)))
print(f"recipes in ≥1 shortlisted cuisine: {(n_cuisines >= 1).sum():,}")
print(f"dropped (≥2 shortlisted cuisines): {(n_cuisines >= 2).sum():,}")
print(f"kept: {len(recipes):,}")
recipes["cuisine"].value_counts().to_frame("recipes")

recipes in ≥1 shortlisted cuisine: 27,160
dropped (≥2 shortlisted cuisines): 431
kept: 26,729


,recipes
cuisine,
italian,7218
mexican,6657
indian,2658
french,2186
greek,2182
middle-eastern,1939
chinese,1927
thai,1172
japanese,790


## 3. Clean the recipes
Principle: **a recipe is only removed if its ingredient list is unusable**, because ingredients are what the networks are built from. Implausible values in other columns (cooking time, nutrition) are set to missing (`NaN`) instead, so we do not lose good recipes because of one bad number.

### 3.1 Recipes with fewer than 2 ingredients
A recipe with one ingredient has no ingredient pairs, so it adds nothing to the co-occurrence network.

In [5]:
few = recipes["ingredients"].map(len) < 2
display(recipes.loc[few, ["id", "name", "cuisine", "ingredients"]])
recipes = recipes[~few]
log.append(("drop recipes with < 2 ingredients", int(few.sum()), len(recipes)))

,id,name,cuisine,ingredients
106735,376622,homemade panko japanese bread crumbs,japanese,[white bread]


### 3.2 Duplicate recipes
Notebook 01 found no repeated ids, but some recipes may have been posted twice. We call two recipes duplicates if they have the same name **and** the same set of ingredients, and keep the first one posted.

In [6]:
recipes = recipes.sort_values(["submitted", "id"])  # so that keep="first" keeps the earliest post
key = recipes["name"].fillna("") + "||" + recipes["ingredients"].map(lambda l: "|".join(sorted(set(l))))
dupe = key.duplicated(keep="first")
display(recipes.loc[key.duplicated(keep=False), ["id", "name", "cuisine", "submitted", "n_ingredients"]].sort_values("name").head(8))
recipes = recipes[~dupe].sort_index()
log.append(("drop duplicate recipes (same name + same ingredients)", int(dupe.sum()), len(recipes)))
print("duplicates dropped:", int(dupe.sum()))

,id,name,cuisine,submitted,n_ingredients
111208,292756,indian style green beans,indian,2008-03-19,9
111141,426090,indian style green beans,indian,2010-05-17,9


duplicates dropped: 1


### 3.3 Cooking time (`minutes`)
* `minutes == 0` cannot be a real cooking time; it means the time was not filled in → `NaN`.
* Very large values: we use a standard outlier rule (Tukey's fence) on the **log** of the time, because times are heavily skewed: a value is an outlier if it is above Q3 + 3 × IQR of log10(minutes). We only use the upper fence; short times like 1–2 minutes are normal (drinks, dressings).

Some flagged recipes are genuinely slow (preserves, liqueurs), so we only blank the value; the recipe stays.

In [7]:
def upper_fence_log10(s, k=3.0):
    x = np.log10(s[s > 0])
    q1, q3 = x.quantile([0.25, 0.75])
    return 10 ** (q3 + k * (q3 - q1))

minutes_fence = upper_fence_log10(recipes["minutes"])
zero_min = recipes["minutes"] == 0
long_min = recipes["minutes"] > minutes_fence
print(f"upper fence: {minutes_fence:,.0f} minutes (≈ {minutes_fence / 1440:.1f} days)")
print(f"minutes == 0 → NaN: {zero_min.sum()} | minutes > fence → NaN: {long_min.sum()}")
display(recipes.loc[long_min, ["name", "cuisine", "minutes"]].sort_values("minutes", ascending=False).head(8))

recipes["minutes"] = recipes["minutes"].astype("float").mask(zero_min | long_min)

upper fence: 1,142 minutes (≈ 0.8 days)
minutes == 0 → NaN: 128 | minutes > fence → NaN: 258


,name,cuisine,minutes
170999,raspberry liqueur,french,172810
145524,nut wine,french,132495
124353,limoncello lemoncella,italian,115230
124351,limoncello italian lemoncello or orangecello,italian,86400
84832,flavored vinegar,french,64815
8529,armenian basterma dried cured beef,middle-eastern,47520
128756,make your own vanilla extract,mexican,43200
36689,carquinyolis almond biscotti,french,21630


### 3.4 Nutrition
The 7 nutrition numbers (calories, then % daily value of total fat, sugar, sodium, protein, saturated fat and carbohydrates) are computed together by Food.com, so if the calories are implausible, the other six are not trustworthy either. We apply the same upper fence to calories and blank all 7 values of a flagged recipe. Low values are kept (water, tea and spice mixes really have ~0 calories).

In [8]:
NUTRITION = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv", "protein_pdv", "sat_fat_pdv", "carbs_pdv"]
recipes[NUTRITION] = pd.DataFrame(recipes["nutrition"].tolist(), columns=NUTRITION, index=recipes.index)

cal_fence = upper_fence_log10(recipes["calories"])
bad_cal = recipes["calories"] > cal_fence
print(f"upper fence: {cal_fence:,.0f} calories | recipes flagged → nutrition set to NaN: {bad_cal.sum()}")
display(recipes.loc[bad_cal, ["name", "cuisine", "calories"]].sort_values("calories", ascending=False).head(8))
recipes.loc[bad_cal, NUTRITION] = np.nan

upper fence: 13,790 calories | recipes flagged → nutrition set to NaN: 2


,name,cuisine,calories
102405,harold s florentine salad dressing,italian,19396.8
134453,mexican ojarascas cookies,mexican,19380.6


In [9]:
recipes[["n_ingredients", "n_steps", "minutes", "calories"]].describe(percentiles=[0.5, 0.99]).T.round(1)

,count,mean,std,min,50%,99%,max
n_ingredients,26727.0,10.3,4.1,2.0,10.0,22.0,38.0
n_steps,26727.0,11.0,6.5,1.0,10.0,33.0,108.0
minutes,26341.0,63.6,90.0,1.0,40.0,500.0,1140.0
calories,26725.0,445.5,506.7,0.0,341.1,2204.4,13171.5


## 4. Clean the reviews
We keep only reviews of the recipes selected above and then remove, step by step:

1. **Reviews written for community cooking games** (Zaar World Tour, Pick A Chef, Recipe Swap, the many *Tag* games, ...). They are written to score points in a game rather than as ordinary feedback, and Zaar World Tour assigns players to foreign cuisines on purpose (notebook 01). The pattern list comes from reading the most common "Made for ..." phrases.
2. **Replies by the recipe's own author.** They are not reviews of the recipe.
3. **Reviews with no text.**

We also set **rating 0 to missing**: on Food.com 0 means "no stars given", not "the worst recipe" (notebook 01). The review text is kept.

In [10]:
reviews = reviews_raw.merge(recipes[["id", "cuisine", "contributor_id"]], left_on="recipe_id", right_on="id", how="inner").drop(columns="id")
rlog = [("reviews of the selected recipes", None, len(reviews))]
print(f"reviews of the selected recipes: {len(reviews):,}")

reviews of the selected recipes: 145,850

**Game reviews.** Two words need care because they also appear in normal reviews: *swap* ("swap the sugar for honey") and *tag* ("price tag", "tag along"). We exclude those uses.

In [11]:
GAME_PATTERNS = {
    "Zaar World Tour":      r"\bzwt ?\d*\b|\bwzt ?\d*\b|zaar world tour|\bworld tour ?\d+|\bwt ?\d\b",
    "Pick A Chef":          r"\bpac\b|pick[ -]a[ -]chef",
    "Recipe Swap":          r"\bswap\b(?!\s+(?:out|around|in|for|it|the|them|with|that|this|some|a|an|to)\b)",
    "Tag games":            r"(?<!price )\btag(?:ged)?\b(?! along)|\bprmrt?\b",
    "Culinary Quest":       r"culinary quest|\bcq ?\d+\b",
    "1-2-3 Hit Wonders":    r"hit wonders|\b1-?2-?3 hits\b",
    "Zaar Stars":           r"zaar stars?\b",
    "My 3 Chefs":           r"\bmy (?:3|three) chefs\b",
    "other named games":    r"please review my recipe|everyday is a holiday|comfort cafe|all you can cook buffet|"
                            r"healthy choices abc|bargain basement|new kids on the block|zaar chef alphabet soup",
}
text = reviews["review"].fillna("").str.lower()
game_hits = pd.DataFrame({name: text.str.contains(p, regex=True) for name, p in GAME_PATTERNS.items()})
is_game = game_hits.any(axis=1)
display(game_hits.sum().rename("reviews matched").to_frame())
print(f"game reviews: {is_game.sum():,} ({is_game.mean():.1%} of the selected reviews)")

,reviews matched
Zaar World Tour,5406
Pick A Chef,1847
Recipe Swap,1046
Tag games,3772
Culinary Quest,364
1-2-3 Hit Wonders,189
Zaar Stars,141
My 3 Chefs,139
other named games,737


game reviews: 12,981 (8.9% of the selected reviews)


In [12]:
# Spot check: 10 random removed reviews, cut around the matching words
GAME_ANY = re.compile("|".join(GAME_PATTERNS.values()))
for t in reviews.loc[is_game, "review"].sample(10, random_state=RNG_SEED):
    m = GAME_ANY.search(t.lower())
    start = max(0, m.start() - 70)
    print("…" + t[start:m.end() + 50].replace("\n", " ") + "…")

… first drawn to this recipe by Dreamer in Ontario's photograph in the PRMR thread and WOW!  It was just so totally delicious…
…d fresh mozzarella.  Thanks MizEmerilLagasse :)  Made for Newest Zaar tag game…
…ad!  It was very refreshing and colorful.  Thanks for sharing!  Veg*n Swap 25…
…ed ingredient, in addition to, the parmesan cheese.  Made for the FYC tag game.…
… liked the zip of the mint.  Thanks for sharing the recipe!  Made for ZWT9, The Apron String Travelers…
…e who liked it more.  Thanks Littlemafia :)  Made for Sweet tradition tag game…
…I made this last night for ZWT2, Canada. YUM!! Ohhh... I ate myself into a comfor…
…Made for Zaar Alphabet Soup Tag game.  This was very very good.  I cut the recipe…
…lly be at least 4 stars as written.  Thank you for posting.  Made for ZWT5 for the Groovy GastroGnomes!…
 Made for Spring PAC 08'…


In [13]:
print("game reviews by cuisine (% of that cuisine's reviews):")
display((is_game.groupby(reviews["cuisine"]).mean() * 100).round(1).sort_values(ascending=False).to_frame("% game"))

reviews = reviews[~is_game]
rlog.append(("drop game reviews", int(is_game.sum()), len(reviews)))

own = reviews["user_id"] == reviews["contributor_id"]
reviews = reviews[~own]
rlog.append(("drop replies by the recipe's author", int(own.sum()), len(reviews)))

empty = reviews["review"].fillna("").str.strip() == ""
reviews = reviews[~empty]
rlog.append(("drop reviews without text", int(empty.sum()), len(reviews)))

zero = reviews["rating"] == 0
reviews["rating"] = reviews["rating"].astype("Int64").mask(zero)
print(f"author replies dropped: {own.sum():,} | empty reviews dropped: {empty.sum():,} | rating 0 → missing: {zero.sum():,}")
print(f"reviews left: {len(reviews):,}")

game reviews by cuisine (% of that cuisine's reviews):


,% game
cuisine,
greek,15.3
french,14.0
middle-eastern,13.0
indian,10.3
thai,9.0
mexican,7.8
italian,6.7
japanese,6.7
chinese,6.3


author replies dropped: 107 | empty reviews dropped: 18 | rating 0 → missing: 7,119
reviews left: 132,744


Recipes whose only reviews were removed stay in `recipes.csv` (the networks do not need reviews); they simply have `n_reviews = 0`.

## 5. Final datasets
We keep the columns that the two parts of the project need. The raw `tags` column is **not** kept, because it contains the sub-cuisine tags; `cuisine` is the only cuisine column.

In [14]:
recipes["n_reviews"] = recipes["id"].map(reviews["recipe_id"].value_counts()).fillna(0).astype(int)

RECIPE_COLS = ["recipe_id", "name", "cuisine", "ingredients", "n_ingredients", "steps", "n_steps",
               "description", "minutes", "submitted", "contributor_id", "n_reviews"] + NUTRITION
REVIEW_COLS = ["review_id", "recipe_id", "cuisine", "user_id", "date", "rating", "review"]

recipes_final = (recipes.rename(columns={"id": "recipe_id"})[RECIPE_COLS]
                 .sort_values(["cuisine", "recipe_id"]).reset_index(drop=True))
reviews_final = reviews[REVIEW_COLS].sort_values(["cuisine", "recipe_id", "date"]).reset_index(drop=True)
print(f"recipes: {len(recipes_final):,} | reviews: {len(reviews_final):,}")
recipes_final.head(3)

recipes: 26,727 | reviews: 132,744


,recipe_id,name,cuisine,ingredients,n_ingredients,steps,n_steps,description,minutes,submitted,contributor_id,n_reviews,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,sat_fat_pdv,carbs_pdv
0,195,chinese hot and sour chicken noodle salad,chinese,"[chicken breast, dry linguine, peppercorn, dried hot red chile, salad oil, rice vinega...",9,"[in a 5-6 quart pan , bring about 3 quarts of water to a boil, add chicken , cover and...",22,originally from,80.0,1999-08-27,1545,1,340.7,19.0,4.0,15.0,31.0,12.0,14.0
1,198,chinese hot and sour pork soup,chinese,"[chicken stock, lean pork, garlic and red chile paste, soy sauce, white pepper, egg su...",17,"[bring stock to a simmer, add soy sauce , pork , mushrooms and chile paste, simmer for...",11,edited recipe from other viewers recomendations,40.0,1999-09-18,1545,49,368.7,20.0,41.0,60.0,63.0,14.0,10.0
2,203,chinese plum sauce,chinese,"[plums, onion, water, gingerroot, garlic, granulated sugar, rice vinegar, coriander, s...",12,"[in large heavy saucepan , bring plums , onions , water , ginger and garlic to boil ov...",9,chinese plum sauce serve this with egg rolls or brush it over chicken and ribs on the...,115.0,1999-08-06,1533,16,316.1,1.0,287.0,12.0,5.0,0.0,26.0


In [15]:
reviews_final.head(3)

,review_id,recipe_id,cuisine,user_id,date,rating,review
0,623253,195,chinese,868654,2010-06-12,5,Fairly simple to make and the outcome is a delicious salad that makes you want another...
1,671682,198,chinese,5067,2001-01-16,5,Great recipe for anyone who like it as close as it come to the real thing! Very simple...
2,671683,198,chinese,5227,2001-01-23,4,my family like the bamboo shoots whole. but we love your recipe


## 6. Mini dataset
For each cuisine separately, we keep a random 30% of its recipes (so 70% are removed), with a fixed random seed so that everyone gets the same mini dataset. Sampling within each cuisine keeps the cuisine proportions the same as in the full dataset. The mini reviews are simply the reviews of the mini recipes.

In [16]:
recipes_mini = (recipes_final.groupby("cuisine", group_keys=False)
                .sample(frac=MINI_KEEP_FRACTION, random_state=RNG_SEED)
                .sort_values(["cuisine", "recipe_id"]).reset_index(drop=True))
reviews_mini = reviews_final[reviews_final["recipe_id"].isin(recipes_mini["recipe_id"])].reset_index(drop=True)
print(f"mini recipes: {len(recipes_mini):,} | mini reviews: {len(reviews_mini):,}")

mini recipes: 8,019 | mini reviews: 38,178


In [17]:
def per_cuisine(rec, rev):
    return pd.DataFrame({
        "recipes": rec["cuisine"].value_counts(),
        "reviews": rev["cuisine"].value_counts(),
        "distinct ingredients": rec.explode("ingredients").groupby("cuisine")["ingredients"].nunique(),
    })

full_t, mini_t = per_cuisine(recipes_final, reviews_final), per_cuisine(recipes_mini, reviews_mini)
comparison = pd.concat({"full": full_t, "mini": mini_t}, axis=1)
comparison[("mini", "% of full recipes")] = (mini_t["recipes"] / full_t["recipes"] * 100).round(1)
comparison = comparison.sort_values(("full", "recipes"), ascending=False)
comparison.loc["TOTAL"] = [
    len(recipes_final), len(reviews_final), recipes_final["ingredients"].explode().nunique(),
    len(recipes_mini), len(reviews_mini), recipes_mini["ingredients"].explode().nunique(),
    round(len(recipes_mini) / len(recipes_final) * 100, 1),
]
comparison

full                                 mini          \
               recipes reviews distinct ingredients recipes reviews   
cuisine                                                               
italian           7218   40272                 3657    2165   13291   
mexican           6657   36868                 3456    1997    9820   
indian            2657    9334                 2024     797    2661   
french            2186    8347                 2383     656    2686   
greek             2182   10830                 1907     655    2429   
middle-eastern    1939    7064                 1826     582    2122   
chinese           1927   10966                 1810     578    2766   
thai              1172    4922                 1428     352    1396   
japanese           789    4141                 1125     237    1007   
TOTAL            26727  132744                 7296    8019   38178   

                                                       
               distinct ingredients % of full recipes  
cuisine                                                
italian                        2215              30.0  
mexican                        2039              30.0  
indian                         1189              30.0  
french                         1311              30.0  
greek                          1046              30.0  
middle-eastern                 1017              30.0  
chinese                        1041              30.0  
thai                            846              30.0  
japanese                        599              30.0  
TOTAL                          4827              30.0

## 7. Sanity checks
These `assert`s stop the notebook if any promise about the datasets is broken.

In [18]:
for name, rec, rev in [("full", recipes_final, reviews_final), ("mini", recipes_mini, reviews_mini)]:
    assert rec["recipe_id"].is_unique, name
    assert rev["review_id"].is_unique, name
    assert set(rec["cuisine"]) == set(SHORTLIST), name                       # all 9 cuisines, nothing else
    assert rev["recipe_id"].isin(rec["recipe_id"]).all(), name               # every review has its recipe
    assert (rev["cuisine"] == rev["recipe_id"].map(rec.set_index("recipe_id")["cuisine"])).all(), name
    assert (rec["ingredients"].map(len) >= 2).all(), name
    assert (rev["review"].str.strip() != "").all(), name
    assert not (rev["rating"] == 0).any(), name
    assert not rev["review"].str.lower().str.contains(GAME_ANY).any(), name  # no game reviews left
    assert (rec["minutes"].dropna() > 0).all(), name
assert recipes_mini["recipe_id"].isin(recipes_final["recipe_id"]).all()      # mini ⊂ full
assert reviews_mini["review_id"].isin(reviews_final["review_id"]).all()
share = recipes_mini["cuisine"].value_counts() / recipes_final["cuisine"].value_counts()
assert share.between(0.29, 0.31).all(), share                               # ~30% of every cuisine
print("all checks passed")

all checks passed


## 8. Export
CSV files, so they open anywhere. The list columns `ingredients` and `steps` are written as **JSON** lists, which can be read back with `json.loads` (or `ast.literal_eval`, as with the raw files). `data/processed/` is git-ignored like the rest of `data/`.

In [19]:
def export(df, name):
    df = df.copy()
    for col in ["ingredients", "steps"]:
        if col in df:
            df[col] = df[col].map(json.dumps)
    path = OUT / f"{name}.csv"
    df.to_csv(path, index=False)
    return {"file": f"data/processed/{name}.csv", "rows": len(df), "MB": round(path.stat().st_size / 1e6, 1)}

files = pd.DataFrame([export(recipes_final, "recipes"), export(reviews_final, "reviews"),
                      export(recipes_mini, "recipes_mini"), export(reviews_mini, "reviews_mini")])
raw_mb = round(sum((RAW / f).stat().st_size for f in ["RAW_recipes.csv", "RAW_interactions.csv"]) / 1e6, 1)
print(f"raw files together: {raw_mb} MB")
files

raw files together: 644.0 MB


,file,rows,MB
0,data/processed/recipes.csv,26727,32.2
1,data/processed/reviews.csv,132744,43.0
2,data/processed/recipes_mini.csv,8019,9.6
3,data/processed/reviews_mini.csv,38178,12.4


**How to load them** (for example in a teammate's notebook):

In [20]:
def load(name):
    df = pd.read_csv(OUT / f"{name}.csv")
    for col in ["submitted", "date"]:
        if col in df:
            df[col] = pd.to_datetime(df[col])
    for col in ["ingredients", "steps"]:
        if col in df:
            df[col] = df[col].map(json.loads)
    if "rating" in df:
        df["rating"] = df["rating"].astype("Int64")
    return df

# Round trip: what we read back must equal what we wrote
for name, df in [("recipes", recipes_final), ("reviews", reviews_final),
                 ("recipes_mini", recipes_mini), ("reviews_mini", reviews_mini)]:
    pd.testing.assert_frame_equal(load(name), df, check_dtype=False)
print("read-back check passed for all 4 files")
load("recipes_mini")[["recipe_id", "name", "cuisine", "ingredients"]].head(3)

read-back check passed for all 4 files


,recipe_id,name,cuisine,ingredients
0,203,chinese plum sauce,chinese,"[plums, onion, water, gingerroot, garlic, granulated sugar, rice vinegar, coriander, s..."
1,3516,general tso s chicken ii,chinese,"[soy sauce, rice vinegar, water, cornstarch, dry sherry, sesame oil, garlic, fresh gin..."
2,4729,wonton soup iii,chinese,"[raw shrimp, lean pork, water chestnuts, green onions, cornstarch, salt, sesame oil, w..."


## 9. Summary of the cleaning

In [21]:
summary = pd.DataFrame(log + rlog, columns=["step", "removed", "left"])
summary.insert(0, "table", ["recipes"] * len(log) + ["reviews"] * len(rlog))
summary["removed"] = summary["removed"].astype("Int64")
summary

,table,step,removed,left
0,recipes,recipes in a shortlisted cuisine,<NA>,27160
1,recipes,drop recipes in two or more shortlisted cuisines,431,26729
2,recipes,drop recipes with < 2 ingredients,1,26728
3,recipes,drop duplicate recipes (same name + same ingredients),1,26727
4,reviews,reviews of the selected recipes,<NA>,145850
5,reviews,drop game reviews,12981,132869
6,reviews,drop replies by the recipe's author,107,132762
7,reviews,drop reviews without text,18,132744
